In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""TFT v2 + LightGBM 앙상블 메타모델
전략:
  1. TFT v2로 상승 확률 예측 (시퀀스 기반)
  2. LightGBM으로 상승 확률 예측 (정형 피처 기반)
  3. 두 확률을 메타모델(LogisticRegression/Stacking)로 결합
"""
import warnings, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
pl.seed_everything(42)
print(f"PyTorch: {torch.__version__}, Lightning: {pl.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
LGBM_FEATURE_PATH = BASE_PATH / "processed" / "lgbm_features" / "lgbm_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_tft_lgbm"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TRAIN_END = "2024-06-30"
VAL_START, VAL_END = "2024-07-01", "2024-12-31"
TEST_START = "2025-01-01"

MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

print("설정 완료")

In [ ]:
# ===== 데이터 로드 =====
# TFT 피처
df_tft = pd.read_parquet(str(TFT_FEATURE_PATH))
df_tft["date"] = pd.to_datetime(df_tft["date"])
df_tft["target_5d"] = df_tft["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df_tft.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"TFT 피처: {len(df_tft):,} rows")

# LightGBM 피처 (있으면 로드)
lgbm_available = False
if LGBM_FEATURE_PATH.exists():
    df_lgbm = pd.read_parquet(str(LGBM_FEATURE_PATH))
    if isinstance(df_lgbm.index, pd.MultiIndex):
        df_lgbm = df_lgbm.reset_index()
    if "date" in df_lgbm.columns:
        df_lgbm["date"] = pd.to_datetime(df_lgbm["date"])
    lgbm_available = True
    print(f"LightGBM 피처: {len(df_lgbm):,} rows, {len(df_lgbm.columns)} cols")
else:
    print(f"LightGBM 피처 없음 ({LGBM_FEATURE_PATH})")
    print("TFT 8피처로 LightGBM도 학습합니다 (동일 데이터, 다른 모델)")

In [ ]:
# ===== TFT v2 모델 정의 =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): x,y=b; l=self.loss_fn(self(x),y); self.log("train_loss",l,prog_bar=True); return l
    def validation_step(self,b,_):
        x,y=b; lo=self(x); self.log("val_loss",self.loss_fn(lo,y),prog_bar=True)
    def configure_optimizers(self):
        o=torch.optim.AdamW(self.parameters(),lr=self.lr,weight_decay=1e-4)
        s=torch.optim.lr_scheduler.ReduceLROnPlateau(o,"min",0.5,patience=5,min_lr=1e-6)
        return {"optimizer":o,"lr_scheduler":{"scheduler":s,"monitor":"val_loss"}}

print("TFT v2 모델 정의 완료")

In [ ]:
# ===== Step 1: TFT v2 체크포인트 로드 or 학습 =====
def make_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

# 체크포인트 탐색
CKPT_PATHS = [
    BASE_PATH / "models" / "tft_v2" / "best.ckpt",
    BASE_PATH / "models" / "tft_v2_backtest" / "best.ckpt",
    MODEL_SAVE_PATH / "tft_best.ckpt",
]
ckpt_found = None
for p in CKPT_PATHS:
    if p.exists():
        ckpt_found = p
        break

if ckpt_found is not None:
    print(f"TFT 체크포인트 로드: {ckpt_found}")
    best_tft = TFTv2.load_from_checkpoint(str(ckpt_found), strict=False)
    best_tft.eval(); best_tft.cuda()
    print("로드 완료 (학습 스킵)")
else:
    print("TFT 체크포인트 없음 -> 신규 학습...")
    train_samples, _ = make_samples(df_tft, "2008-01-01", TRAIN_END, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    labels = [s[1] for s in train_samples]
    n0=sum(1 for l in labels if l==0); n1=sum(1 for l in labels if l==1)
    cw = torch.tensor([len(labels)/(2*max(n0,1)), len(labels)/(2*max(n1,1))], dtype=torch.float32)

    model_tft = TFTv2(N_FEATURES, MAX_ENCODER_LENGTH, 128, 4, 1, 0.2, 2, 5e-4, cw)
    es = EarlyStopping(monitor="val_loss", patience=8, mode="min", verbose=False)
    val_size = max(len(train_samples)//10, 1000)
    trainer = pl.Trainer(max_epochs=50, accelerator="gpu", devices=1, gradient_clip_val=0.1,
        callbacks=[es], enable_progress_bar=True, log_every_n_steps=50)
    trainer.fit(model_tft,
        train_dataloaders=DataLoader(SeqDS(train_samples[:-val_size]), batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
        val_dataloaders=DataLoader(SeqDS(train_samples[-val_size:]), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0))

    best_tft = TFTv2.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, strict=False)
    best_tft.eval(); best_tft.cuda()
    # 저장
    save_path = MODEL_SAVE_PATH / "tft_best.ckpt"
    trainer.save_checkpoint(str(save_path))
    print(f"학습 완료 & 저장: {save_path}")

# 예측 생성
val_samples, val_metas = make_samples(df_tft, VAL_START, VAL_END, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
test_samples, test_metas = make_samples(df_tft, TEST_START, None, MAX_ENCODER_LENGTH, CLEANED_FEATURES)

def predict_tft(model, samples):
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

tft_val_probs, val_labels = predict_tft(best_tft, val_samples)
tft_test_probs, test_labels = predict_tft(best_tft, test_samples)
print(f"TFT 예측 완료: val={len(tft_val_probs):,}, test={len(tft_test_probs):,}")

In [ ]:
# ===== Step 2: LightGBM 학습 & 예측 =====
# TFT와 동일한 피처를 사용하되, 시퀀스가 아닌 단일 시점 피처로
# (마지막 시점의 피처값만 사용)

def make_lgbm_data(full_df, start, end, feats):
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask].copy()
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values.astype(int)
    return X, y

X_train, y_train = make_lgbm_data(df_tft, "2008-01-01", TRAIN_END, CLEANED_FEATURES)
X_val, y_val = make_lgbm_data(df_tft, VAL_START, VAL_END, CLEANED_FEATURES)
X_test, y_test = make_lgbm_data(df_tft, TEST_START, None, CLEANED_FEATURES)

print(f"LightGBM 데이터: train={len(X_train):,}, val={len(X_val):,}, test={len(X_test):,}")

# LightGBM 학습
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    class_weight="balanced", random_state=42, verbose=-1)

lgbm_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
               callbacks=[lgb.early_stopping(50, verbose=False)])

lgbm_val_probs = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_test_probs = lgbm_model.predict_proba(X_test)[:, 1]
print(f"LightGBM 예측 완료")

In [ ]:
# ===== Step 3: 예측 정렬 =====
# TFT는 시퀀스(30일 이후부터), LightGBM은 모든 행
# 날짜+종목 기준으로 매칭 필요

# TFT 예측을 DataFrame으로
tft_val_df = pd.DataFrame(val_metas)
tft_val_df["tft_prob"] = tft_val_probs
tft_val_df["label"] = val_labels

tft_test_df = pd.DataFrame(test_metas)
tft_test_df["tft_prob"] = tft_test_probs
tft_test_df["label"] = test_labels

# LightGBM 예측을 DataFrame으로
val_mask = (df_tft["date"]>=VAL_START) & (df_tft["date"]<=VAL_END)
lgbm_val_df = df_tft[val_mask][["date","ticker"]].copy().reset_index(drop=True)
lgbm_val_df["lgbm_prob"] = lgbm_val_probs
lgbm_val_df["date"] = lgbm_val_df["date"].astype(str).str[:10]

test_mask = df_tft["date"]>=TEST_START
lgbm_test_df = df_tft[test_mask][["date","ticker"]].copy().reset_index(drop=True)
lgbm_test_df["lgbm_prob"] = lgbm_test_probs
lgbm_test_df["date"] = lgbm_test_df["date"].astype(str).str[:10]

# 매칭
val_merged = tft_val_df.merge(lgbm_val_df, on=["date","ticker"], how="inner")
test_merged = tft_test_df.merge(lgbm_test_df, on=["date","ticker"], how="inner")
print(f"매칭 완료: val={len(val_merged):,}, test={len(test_merged):,}")

In [ ]:
# ===== Step 4: 앙상블 메타모델 =====
# 방법 1: Simple Average
val_avg = (val_merged["tft_prob"] + val_merged["lgbm_prob"]) / 2
test_avg = (test_merged["tft_prob"] + test_merged["lgbm_prob"]) / 2

# 방법 2: Weighted Average (TFT에 더 높은 가중치)
val_wavg = val_merged["tft_prob"] * 0.6 + val_merged["lgbm_prob"] * 0.4
test_wavg = test_merged["tft_prob"] * 0.6 + test_merged["lgbm_prob"] * 0.4

# 방법 3: Logistic Regression Stacking
X_meta_val = val_merged[["tft_prob", "lgbm_prob"]].values
y_meta_val = val_merged["label"].values
X_meta_test = test_merged[["tft_prob", "lgbm_prob"]].values
y_meta_test = test_merged["label"].values

meta_lr = LogisticRegression(random_state=42, class_weight="balanced")
meta_lr.fit(X_meta_val, y_meta_val)  # val로 학습
test_lr_probs = meta_lr.predict_proba(X_meta_test)[:, 1]

print(f"Meta LR 계수: TFT={meta_lr.coef_[0][0]:.3f}, LightGBM={meta_lr.coef_[0][1]:.3f}")
print(f"Meta LR intercept: {meta_lr.intercept_[0]:.3f}")

In [ ]:
# ===== Step 5: 앙상블 결과 비교 =====
def eval_probs(probs, labels, name):
    preds = (probs >= 0.5).astype(int)
    da = accuracy_score(labels, preds)
    try: auc = roc_auc_score(labels, probs)
    except: auc = float("nan")
    pos = preds.mean()
    print(f"  {name:25s}: DA={da*100:.2f}%, AUC={auc:.4f}, 양성={pos*100:.1f}%")
    return da, auc

print("="*60)
print("  테스트 성능 비교")
print("="*60)
eval_probs(test_merged["tft_prob"].values, y_meta_test, "TFT v2 단독")
eval_probs(test_merged["lgbm_prob"].values, y_meta_test, "LightGBM 단독")
eval_probs(test_avg.values, y_meta_test, "Simple Average")
eval_probs(test_wavg.values, y_meta_test, "Weighted (TFT 0.6)")
eval_probs(test_lr_probs, y_meta_test, "Stacking (LogisticReg)")
print("="*60)

In [ ]:
# ===== 최적 앙상블 상세 =====
# 가장 좋은 방법의 classification report
methods = {
    "Simple Avg": test_avg.values,
    "Weighted": test_wavg.values,
    "Stacking": test_lr_probs,
}
best_name = max(methods, key=lambda k: accuracy_score(y_meta_test, (methods[k]>=0.5).astype(int)))
best_probs = methods[best_name]
best_preds = (best_probs >= 0.5).astype(int)

print(f"\n최적 앙상블: {best_name}")
print(classification_report(y_meta_test, best_preds, target_names=["하락","상승"]))

In [ ]:
# ===== 저장 =====
# 앙상블 예측 저장 (백테스트용)
ensemble_df = test_merged[["date","ticker","label"]].copy()
ensemble_df["tft_prob"] = test_merged["tft_prob"].values
ensemble_df["lgbm_prob"] = test_merged["lgbm_prob"].values
ensemble_df["ensemble_prob"] = best_probs
ensemble_df["ensemble_pred"] = best_preds
ensemble_df.to_parquet(str(MODEL_SAVE_PATH / f"ensemble_predictions_{datetime.now().strftime('%Y%m%d')}.parquet"))

json.dump({"best_method": best_name,
  "tft_da": round(accuracy_score(y_meta_test,(test_merged['tft_prob']>=0.5).astype(int))*100,2),
  "lgbm_da": round(accuracy_score(y_meta_test,(test_merged['lgbm_prob']>=0.5).astype(int))*100,2),
  "ensemble_da": round(accuracy_score(y_meta_test,best_preds)*100,2)},
  open(str(MODEL_SAVE_PATH/"metrics.json"),"w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## TFT v2 + LightGBM 앙상블\n\n### 앙상블 방법\n1. **Simple Average**: (TFT + LightGBM) / 2\n2. **Weighted Average**: TFT*0.6 + LightGBM*0.4\n3. **Stacking**: LogisticRegression 메타모델\n\n### 기대효과\n- TFT: 시퀀스 패턴 (시간축 정보)\n- LightGBM: 단일 시점 피처 (비선형 관계)\n- 서로 다른 관점 → 앙상블 시 다양성 확보